# Notebook 2 — Gold Layer (Spark SQL)
**Project:** Industrial Sensor Pipeline (NASA CMAPSS)

This notebook covers:
- Reading validated Silver data
- Registering as a Spark SQL temp view
- Building Gold analytics views:
  - Rolling sensor averages (degradation trends)
  - Engine health summary
  - Sensor correlation per engine
  - Feature engineering for ML (rolling stats, rate of change)

**Why Spark SQL here?**
In Microsoft Fabric and Databricks, analysts write SQL in notebooks against
distributed data. This mirrors that pattern exactly.

## Setup

In [ ]:
!pip install pyspark -q

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import os

spark = SparkSession.builder \
    .appName("IndustrialSensorPipeline_Gold") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")

## Load Silver & Register as SQL View

In [ ]:
# Read Silver Parquet
df_silver = spark.read.parquet("data/silver/sensor_readings_validated")

# Filter to valid rows only — Gold only sees clean data
df_valid = df_silver.filter(F.col("status") == "valid")

# Register as temp view — now we can query it with SQL
df_valid.createOrReplaceTempView("silver_sensors")

print(f"Valid rows available for Gold: {df_valid.count():,}")
print(f"Unique engines: {df_valid.select('engine_id').distinct().count()}")

## Gold View 1 — Rolling Sensor Averages

Rolling averages smooth out noise and reveal degradation trends.
Same concept as moving averages in the finance pipeline — but for engine health.

In [ ]:
gold_rolling = spark.sql("""
    SELECT
        engine_id,
        cycle,
        sensor_02,
        sensor_04,
        sensor_11,

        -- 10-cycle rolling average per engine
        AVG(sensor_02) OVER (
            PARTITION BY engine_id
            ORDER BY cycle
            ROWS BETWEEN 9 PRECEDING AND CURRENT ROW
        ) AS sensor_02_rolling_avg_10,

        AVG(sensor_04) OVER (
            PARTITION BY engine_id
            ORDER BY cycle
            ROWS BETWEEN 9 PRECEDING AND CURRENT ROW
        ) AS sensor_04_rolling_avg_10,

        AVG(sensor_11) OVER (
            PARTITION BY engine_id
            ORDER BY cycle
            ROWS BETWEEN 9 PRECEDING AND CURRENT ROW
        ) AS sensor_11_rolling_avg_10,

        -- 30-cycle rolling average
        AVG(sensor_02) OVER (
            PARTITION BY engine_id
            ORDER BY cycle
            ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
        ) AS sensor_02_rolling_avg_30

    FROM silver_sensors
    ORDER BY engine_id, cycle
""")

gold_rolling.createOrReplaceTempView("gold_rolling_averages")
gold_rolling.show(10)
print(f"Rows in gold_rolling_averages: {gold_rolling.count():,}")

## Gold View 2 — Engine Health Summary

Per-engine summary: how long did each engine last, and what were the
average sensor readings? Useful for comparing engine cohorts.

In [ ]:
gold_health = spark.sql("""
    SELECT
        engine_id,
        MAX(cycle)            AS total_cycles,
        ROUND(AVG(sensor_02), 2) AS avg_sensor_02,
        ROUND(AVG(sensor_03), 2) AS avg_sensor_03,
        ROUND(AVG(sensor_04), 2) AS avg_sensor_04,
        ROUND(AVG(sensor_07), 2) AS avg_sensor_07,
        ROUND(AVG(sensor_11), 2) AS avg_sensor_11,
        ROUND(STDDEV(sensor_02), 3) AS stddev_sensor_02,

        -- Engines with longer life might indicate better operating conditions
        CASE
            WHEN MAX(cycle) > 200 THEN 'long_life'
            WHEN MAX(cycle) > 130 THEN 'medium_life'
            ELSE 'short_life'
        END AS life_category

    FROM silver_sensors
    GROUP BY engine_id
    ORDER BY total_cycles DESC
""")

gold_health.createOrReplaceTempView("gold_engine_health")
gold_health.show(15)
print("\nLife category distribution:")
gold_health.groupBy("life_category").count().show()

## Gold View 3 — Feature Engineering for ML

This is where data engineering meets machine learning.
We create features that an ML model would need to predict Remaining Useful Life (RUL):
- Rolling mean and stddev (signal stability)
- Rate of change (is the sensor trending up or down?)
- Cycle percentage (how far through the engine's life are we?)

In [ ]:
gold_features = spark.sql("""
    SELECT
        engine_id,
        cycle,
        cycle_pct,

        sensor_02,
        sensor_04,
        sensor_11,

        -- Rolling mean (window = 10 cycles)
        ROUND(AVG(sensor_02) OVER (
            PARTITION BY engine_id ORDER BY cycle
            ROWS BETWEEN 9 PRECEDING AND CURRENT ROW
        ), 3) AS s02_mean_10,

        -- Rolling stddev — measures signal instability (early anomaly indicator)
        ROUND(STDDEV(sensor_02) OVER (
            PARTITION BY engine_id ORDER BY cycle
            ROWS BETWEEN 9 PRECEDING AND CURRENT ROW
        ), 4) AS s02_std_10,

        -- Rate of change: current - previous cycle value
        ROUND(sensor_02 - LAG(sensor_02, 1) OVER (
            PARTITION BY engine_id ORDER BY cycle
        ), 4) AS s02_delta,

        ROUND(sensor_04 - LAG(sensor_04, 1) OVER (
            PARTITION BY engine_id ORDER BY cycle
        ), 4) AS s04_delta,

        -- Target variable for ML: RUL = max_cycle - current_cycle
        (max_cycle - cycle) AS rul

    FROM silver_sensors
    ORDER BY engine_id, cycle
""")

gold_features.createOrReplaceTempView("gold_ml_features")
gold_features.show(10)
print(f"\nFeature table rows: {gold_features.count():,}")

In [ ]:
# Quick sanity check: RUL should decrease as cycle increases
spark.sql("""
    SELECT engine_id, cycle, rul
    FROM gold_ml_features
    WHERE engine_id = 1
    ORDER BY cycle
    LIMIT 10
""").show()

## Gold View 4 — Late-Life Sensor Drift

Compare average sensor readings in early life (first 30 cycles) vs late life
(last 30 cycles). Drift indicates degradation — useful for predictive maintenance.

In [ ]:
gold_drift = spark.sql("""
    WITH early AS (
        SELECT engine_id,
            AVG(sensor_02) AS s02_early,
            AVG(sensor_04) AS s04_early,
            AVG(sensor_11) AS s11_early
        FROM silver_sensors
        WHERE cycle <= 30
        GROUP BY engine_id
    ),
    late AS (
        SELECT engine_id,
            AVG(sensor_02) AS s02_late,
            AVG(sensor_04) AS s04_late,
            AVG(sensor_11) AS s11_late
        FROM silver_sensors
        WHERE rul <= 30
        GROUP BY engine_id
    )
    SELECT
        e.engine_id,
        ROUND(l.s02_late - e.s02_early, 3) AS s02_drift,
        ROUND(l.s04_late - e.s04_early, 3) AS s04_drift,
        ROUND(l.s11_late - e.s11_early, 3) AS s11_drift
    FROM early e
    JOIN late l ON e.engine_id = l.engine_id
    ORDER BY ABS(s02_drift) DESC
""")

gold_drift.createOrReplaceTempView("gold_sensor_drift")
gold_drift.show(15)
print("\nPositive drift = sensor reading increased toward end of life")

## Write Gold Layer to Parquet

In [ ]:
os.makedirs("data/gold", exist_ok=True)

gold_rolling.write.mode("overwrite").parquet("data/gold/rolling_averages")
gold_health.write.mode("overwrite").parquet("data/gold/engine_health")
gold_features.write.mode("overwrite").parquet("data/gold/ml_features")
gold_drift.write.mode("overwrite").parquet("data/gold/sensor_drift")

print("All Gold tables written:")
print("  data/gold/rolling_averages")
print("  data/gold/engine_health")
print("  data/gold/ml_features       ← input for Notebook 3 (anomaly detection)")
print("  data/gold/sensor_drift")

## Summary

| Gold Table | Rows | Key columns | Use case |
|---|---|---|---|
| `rolling_averages` | ~20k | rolling_avg_10, rolling_avg_30 | Trend visualization |
| `engine_health` | 100 | total_cycles, life_category | Fleet overview |
| `ml_features` | ~20k | rul, s02_mean_10, s02_std_10, deltas | ML model input |
| `sensor_drift` | 100 | s02/s04/s11 drift | Degradation analysis |

**Next:** Notebook 3 — Anomaly detection on sensor data using rolling statistics